<b>  Part - (1) : Develop a neural network based classification network from scratch: </b>  This programming assignment uses MNIST digit dataset. It consists of large collection of handwritten digits from 0 to 9. These images are formated as 28x28 pixel gray scale images. The objective of this programming assignment is to design a neural network architecture that takes input as 28x28 image (or 784 dimensional vector) as input and predicts the digit information in it. Although there are diffrent varieties of neural network architecture to solve this task, this programming assignment uses only the feed forward network.  

<h5> 1. Load MNIST data and create train, test splits </h5>

- The MNIST dataset consists of around 70,000 images. Divide the dataset into two segments: training and testing. Allocate 60,000 images for training and 10,000 images for testing

- Code for downloading the data and creating train-test splits is provided 

<h5> 2. Design a simple classification network </h5>

- Let us use three layer feed-forward neral network. Use 512 nodes in the hidden layers and 10 nodes in the output layer. The output $\textbf{y}$ from the input $\textbf{x}$ is computed as follows

$$ \textbf{y} = h\left(\textbf{W}_{3}g\left(\textbf{W}_{2}g\left(\textbf{W}_{1}\textbf{x}\right)\right)\right) $$

- where $\textbf{W}_{1} \in \mathbb{R}^{512 \times 784}\;$,$\;\textbf{W}_{2} \in \mathbb{R}^{512 \times 512}\;$,$\;\textbf{W}_{3} \in \mathbb{R}^{10 \times 512}\;$ are the parameters of the network. g(.) is the hidden layer activation function. h(.) is the output layer activation function  

- Consider g(.) as ReLU activation function. Softmax activation function should be used at the last layer h(.), to get the posterior probability of the classes.

- Training classification network:

- Flatten the 28x28 images to arrive at 784 dimensional vector.

- Randomly initialize the parameters of network, $\textbf{W}_{1} \in \mathbb{R}^{784 \times 512}\;$,$\;\textbf{W}_{2} \in \mathbb{R}^{512 \times 512}\;$,$\;\textbf{W}_{3} \in \mathcal{R}^{512 \times 10}$  

- Feedforward the batch of input vectors to get the posterior probability of classes.
- Compute the loss between the estimated posterior probabilities and the true targets.
- Update the parameters of network to minimize the loss function.
- Backpropagate the loss function to get the gradients.   
- You can use stochastic gradient descent (SGD) optimization algorithm to update the parameters.
- Cleverly set the hyperparameters involved in this optimization process.

<h5> 3. Evaluate the performance of classification network </h5>

- feed-forward the MNIST data through the trained classification network to get class posteriors.   
- Assign the input to the class having maximum posterior probability 
- Compute the loss and accuaracy 
- Report your observations 






In [1]:
#All imports
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import collections

- $\large{Load\;\;MNIST\;\;data}$

In [2]:
import torchvision.datasets as datasets
mnist_trainset = datasets.MNIST(root='./data', train=True, download=True, transform=None)
mnist_testset = datasets.MNIST(root='./data', train=False, download=True, transform=None)

#Training data
mnist_traindata = mnist_trainset.data.numpy()
mnist_trainlabel = mnist_trainset.targets.numpy()
print("Training data",mnist_traindata.shape)
print("Training labels",mnist_trainlabel.shape)

#Testing data
mnist_testdata = mnist_testset.data.numpy()
mnist_testlabel = mnist_testset.targets.numpy()
print("Testing data",mnist_testdata.shape)
print("Testing labels",mnist_testlabel.shape)

Training data (60000, 28, 28)
Training labels (60000,)
Testing data (10000, 28, 28)
Testing labels (10000,)


- $\large{Define\;\;the\;\;architechture}$

In [3]:
# Complete the below function to impliment ReLU activation function
def ReLu(inp):
    outp = np.maximum(inp, 0) 
    return outp

# Complete the below function to impliment gradient of ReLU activation function
def gradReLu(inp):
    outp = np.where(inp < 0, 0, 1) 
    return outp

# Complete the below function to impliment softmax activation function
def softmax(inp):
    exp_values = np.exp(inp)
    outp = exp_values / np.sum(exp_values, axis=1, keepdims=True)
    return outp

# Complete the below function to impliment forward propagation of data
def fwdPropagate(inputs, bias, weights):
    # Inputs: input data, paramters of network
    W1, W2, W3 = weights    # W1 (512, 784), W2 (512, 512), W3 (10, 512)
    b1, b2, b3 = bias       # b1 (512, 1), b2 (512, 1), b3 (10, 1)
    
    a1 = inputs @ W1.T + b1.T   # (batch_size, 512)
    z1 = ReLu(a1)
    a2 = z1 @ W2.T + b2.T       # (batch_size, 512)
    z2 = ReLu(a2)
    a3 = z2 @ W3.T + b3.T       # (batch_size, 10)
    y = softmax(a3)
    
    return [z1, z2, y, a1, a2]

# Complete the below function to compute the gradients
def computeGradients(inputs, targets, weights, activations):
    # Inputs: input data, targets, parameters of netwrok, intermediate activations
    W1, W2, W3 = weights        # W1 (512, 784), W2 (512, 512), W3 (10, 512)

    # calculate activations
    z1, z2, y, a1, a2 = activations     # a1 (batch_size, 512), a2 (batch_size, 512), y (batch_size, 10)
    
    # compute the loss
    
    # Compute the derivative of loss at parameters
    delta3 = y - targets        # delta3 (batch_size, 10)
    dj_dw3 = delta3.T @ z2     # (10, 512)
    db3_grad = np.sum(delta3, axis = 0, keepdims = True).T    # (10, 1)
    
    # Hidden layer 2
    delta2 = (delta3 @ W3) * gradReLu(a2)        # delta3 (batch_size, 512)
    dj_dw2 = delta2.T @ z1     # (512, 512)
    db2_grad = np.sum(delta2, axis = 0, keepdims = True).T    # (10, 1)
    
    # Hidden layer 1
    delta1 = (delta2 @ W2) * gradReLu(a1)        # delta3 (batch_size, 10)
    dj_dw1 = delta1.T @ inputs     # (512, 784)
    db1_grad = np.sum(delta1, axis = 0, keepdims = True).T    # (512, 1)
    
    return [dj_dw1, dj_dw2, dj_dw3, db1_grad, db2_grad, db3_grad]

# Complete the below function to update the parameters using the above computed gradients
def applyGradients(weights, bias, gradients, learning_rate):
    # Inputs: weights, gradients, and learning rate
    W1, W2, W3 = weights        # W1 (512, 784), W2 (512, 512), W3 (10, 512)
    b1, b2, b3 = bias
    
    # Get the gradients
    dj_dw1, dj_dw2, dj_dw3, db1_grad, db2_grad, db3_grad = gradients
    b1 = b1 - learning_rate * db1_grad
    b2 = b2 - learning_rate * db2_grad
    b3 = b3 - learning_rate * db3_grad
    
    W1 = W1 - learning_rate * dj_dw1
    W2 = W2 - learning_rate * dj_dw2
    W3 = W3 - learning_rate * dj_dw3 
    
    return [W1, W2, W3], [b1, b2, b3]

# Complete the below function to complete the backpropagation ste
def backPropagate(inputs, bias, targets, weights, activations, learning_rate):
    # Inputs: input data, targets, parameters of network, intermediate activations, learning rate of optimization algorithm
    gradients = computeGradients(inputs, targets, weights, activations)
    
    # update weights
    weights, bias = applyGradients(weights, bias, gradients, learning_rate)
    
    return weights, bias

- $\large{Train\;\;the\;\;network}$

In [4]:
def calc_error(y_train, y_pred):
    tot = len(y_train)
    count_match = 0
    for i in range(tot):
        if (y_train[i] == y_pred[i]):
            count_match += 1
    acc = count_match / tot
    err = 1 - acc
    
    return acc, err

In [5]:
# Complete the below function to complete the training of network
def training(inputs, targets, batch_size = 300, epochs = 30, train_val_split = 0.8, learning_rate = 0.0001):
    # set hyperparameters
    hidden_units = 512
    n_classes = 10
    n_samples = inputs.shape[0]
    
    # Split the training data into two parts.
    # Use 90 percent of training data for training the network.
    # Remaining 10 percent as validation data
    split_idx = int(train_val_split * n_samples)
    train_inputs, train_targets = inputs[ : split_idx], targets[ : split_idx]
    test_inputs, test_targets = inputs[split_idx : ], targets[split_idx : ]
    
    n_batches = split_idx // batch_size
    
    # Randomly initialize weights
    W1 = np.random.randn(hidden_units, inputs.shape[1]) * 0.01
    b1 = np.random.uniform(0, 1, (hidden_units, 1))
    W2 = np.random.randn(hidden_units, hidden_units) * 0.01
    b2 = np.random.uniform(0, 1, (hidden_units, 1))
    W3 = np.random.randn(n_classes, hidden_units) * 0.01
    b3 = np.random.uniform(0, 1, (n_classes, 1))
    weights = [W1, W2, W3]
    bias = [b1, b2, b3]
    
    # Interate for epochs times
    for epoch in range(epochs):
        # Shuffle the training data
        idxs = np.random.permutation(split_idx)
        train_inputs_shuffled = train_inputs[idxs]
        train_targets_shuffled = train_targets[idxs]
        
        # Iterate through the batches of data
        for batch in range(n_batches):
            # Get the batch of data
            st = batch * batch_size
            ed = min((batch + 1) * batch_size, split_idx)
            # print(st,ed)
            input_batch = train_inputs_shuffled[st : ed]
            target_batch = train_targets_shuffled[st : ed]
            
            # Forward propagation
            activations = fwdPropagate(input_batch, bias, weights)

            # Backward propagation
            weights, bias = backPropagate(input_batch, bias, target_batch, weights, activations, learning_rate)
            
        # Compute outpus on trianing data
        _, _, train_prob, _, _ = fwdPropagate(train_inputs, bias, weights)
        y1_pred = np.argmax(train_prob, axis = 1)
        y1_train = np.argmax(train_targets, axis = 1)
        train_acc, train_error = calc_error(y1_train, y1_pred)
        
        # Compute outpus on validation data
        _, _, test_prob, _, _ = fwdPropagate(test_inputs, bias, weights)
        y2_pred = np.argmax(test_prob, axis = 1)
        y2_test = np.argmax(test_targets, axis = 1)
        test_acc, test_error = calc_error(y2_test, y2_pred)
        
        # Print the statistics of training
        print(f"Epoch {epoch+1}/{epochs} - Training Error: {train_error:.4f} - Training Accuracy: {train_acc:.4f} - Validation Error: {test_error:.4f} - Validation Accuracy: {test_acc:.4f}") 
        
    return weights, bias


# Reshape the data
x_train_1 = []
x_test_1 = []

for i in range(60000):
    x_train_1.append(mnist_traindata[i].flatten())
    if (i < 10000):
        x_test_1.append(mnist_testdata[i].flatten())
    
x_train_1 = np.array(x_train_1) / 255.0
x_test_1 = np.array(x_test_1) / 255.0

# one-hot encode the labels
y_train_1 = np.eye(10)[mnist_trainlabel]
y_test_1 = np.eye(10)[mnist_testlabel]

print(x_train_1.shape)
print(y_train_1.shape)

print(x_test_1.shape)
print(y_test_1.shape)
 
# Call the training function to train the network
train_weights , train_bias = training(x_train_1, y_train_1)

(60000, 784)
(60000, 10)
(10000, 784)
(10000, 10)
Epoch 1/30 - Training Error: 0.6056 - Training Accuracy: 0.3944 - Validation Error: 0.6053 - Validation Accuracy: 0.3947
Epoch 2/30 - Training Error: 0.4072 - Training Accuracy: 0.5928 - Validation Error: 0.3969 - Validation Accuracy: 0.6031
Epoch 3/30 - Training Error: 0.2653 - Training Accuracy: 0.7347 - Validation Error: 0.2444 - Validation Accuracy: 0.7556
Epoch 4/30 - Training Error: 0.1746 - Training Accuracy: 0.8254 - Validation Error: 0.1620 - Validation Accuracy: 0.8380
Epoch 5/30 - Training Error: 0.1430 - Training Accuracy: 0.8570 - Validation Error: 0.1336 - Validation Accuracy: 0.8664
Epoch 6/30 - Training Error: 0.1269 - Training Accuracy: 0.8731 - Validation Error: 0.1207 - Validation Accuracy: 0.8793
Epoch 7/30 - Training Error: 0.1169 - Training Accuracy: 0.8831 - Validation Error: 0.1132 - Validation Accuracy: 0.8868
Epoch 8/30 - Training Error: 0.1091 - Training Accuracy: 0.8909 - Validation Error: 0.1064 - Validation

- $\large{Evaluate\;\;the\;\;performance\;\;on\;\;test\;\;data}$

In [6]:
def evaluate_performance(inputs, targets, weights, bias):
    # Forward propagate the inputs
    out = fwdPropagate(inputs, bias, weights)[2]
    
    # Assign input to the class having maximum posterior probability
    y_pred = np.argmax(out, axis = 1)
    y_train = np.argmax(targets, axis=1)
    
    # Compute accuracy
    acc, err = calc_error(y_train, y_pred)
    
    return acc

test_accuracy = evaluate_performance(x_test_1, y_test_1, train_weights, train_bias)
print("Test Accuracy:", test_accuracy)

Test Accuracy: 0.9384


<b> Report your observations </b>

1. The training process involves iterating through epochs (30 epochs in this case) and batches of data. Each epoch goes through the entire training dataset in batches.

2. During training, the model's performance is evaluated on both the training and validation datasets. The training error, training accuracy, validation error, and validation accuracy are printed after each epoch.

3. As the epochs progress, both training and validation accuracies increase while errors decrease, indicating that the model is learning and generalizing well.

4. After training, the trained model is evaluated on the test dataset using the evaluate_performance function, which computes the accuracy on the test set. The test accuracy achieved is approximately 93.84%.



<b> Part - (2) : Understanding activation functions: </b> In this part you will learn to use different activation functions for the classification task and compare their performances.

- Train MNIST digit classification network with different activation functions i.e. Sigmoid, Tanh, ReLU, LeakyReLU etc. You can stick to stochastic gradient descent optimization algorithm for this part 

- Report the accuray on MNIST test data for all the experiments. Write down your observations in the report.



- $\large{Train\;\;the\;\;network\;\;with\;\;different\;\;activation\;\;functions}$

In [10]:
# Complete the below function to impliment sigmoid activation function
def sigmoid(inp):
    # inputs : inp -> matrix
    outp = 1 / (1 + np.exp(inp))     
    return outp

# Complete the below function to impliment gradient of sigmoid activation function
def gradSigmoid(inp):
    # inputs : inp -> matrix
    outp = sigmoid(inp) * (np.full(inp.shape, 1) - sigmoid(inp))
    return outp

# Complete the below function to impliment tanh activation function
def tanH(inp):
    # inputs : inp -> matrix
    outp = np.tanh(inp)     
    return outp

# Complete the below function to impliment gradient of tanh activation function
def gradtanH(inp):
    # inputs : inp -> matrix
    outp = np.full(inp.shape, 1) - tanH(inp) * tanH(inp)
    return outp

# Complete the below function to impliment Leaky ReLu activation function
def LeakyReLu(inp, alpha = 0.01):
    # inputs : inp -> matrix
    outp = np.maximum(alpha * inp, inp)     
    return outp

# Complete the below function to impliment gradient of Leaky ReLu activation function
def gradLeakyReLu(inp, alpha = 0.01):
    # inputs : inp -> matrix
    outp = np.where(inp < 0, alpha, 1)
    return outp

In [11]:
# Complete the below function to impliment forward propagation of data
def fwdPropagate_2(inputs, bias, weights, act_func):
    # Inputs: input data, paramters of network
    W1, W2, W3 = weights    # W1 (512, 784), W2 (512, 512), W3 (10, 512)
    b1, b2, b3 = bias
    
    # compute a1
    a1 = inputs @ W1.T + b1.T  # (batch_size, 512)
    
    # compute z1
    z1 = sigmoid(a1)
    if (act_func == "tanh"):
        z1 = tanH(a1)
    elif (act_func == "leakyrelu"):
        z1 = LeakyReLu(a1)
    
    # compute a2  
    a2 = z1 @ W2.T + b2.T    # (batch_size, 512)
    
    # compute z2
    z2 = sigmoid(a2)
    if (act_func == "tanh"):
        z2 = tanH(a2)
    elif (act_func == "leakyrelu"):
        z2 = LeakyReLu(a2)
    
    # compute a3 and y
    a3 = z2 @ W3.T + b3.T      # (batch_size, 10)
    y = softmax(a3)
    
    return [z1, z2, y, a1, a2]

# Complete the below function to compute the gradients
def computeGradients_2(inputs, targets, weights, activations, act_func):
    # Inputs: input data, targets, parameters of netwrok, intermediate activations
    W1, W2, W3 = weights        # W1 (512, 784), W2 (512, 512), W3 (10, 512)

    # calculate activations
    z1, z2, y, a1, a2 = activations     # a1 (batch_size, 512), a2 (batch_size, 512), y (batch_size, 10)
    
    # compute the loss
    
    # Compute the derivative of loss at parameters
    delta3 = y - targets        # delta3 (batch_size, 10)
    dj_dw3 = delta3.T @ z2     # (10, 512)
    db3_grad = np.sum(delta3, axis = 0, keepdims = True).T    # (10, 1)

    # Hidden layer 2
    da2 = gradSigmoid(a2)
    if (act_func == "tanh"):
        da2 = gradtanH(a2)
    elif (act_func == "leakyrelu"):
        da2 = gradLeakyReLu(a2)
        
    delta2 = (delta3 @ W3) * da2        # delta3 (batch_size, 512)
    dj_dw2 = delta2.T @ z1     # (512, 512)
    db2_grad = np.sum(delta2, axis = 0, keepdims = True).T    # (10, 1)
    
    # Hidden layer 1
    da1 = gradSigmoid(a1)
    if (act_func == "tanh"):
        da1 = gradtanH(a1)
    elif (act_func == "leakyrelu"):
        da1 = gradLeakyReLu(a1)
        
    delta1 = (delta2 @ W2) * da1        # delta3 (batch_size, 10)
    dj_dw1 = delta1.T @ inputs     # (512, 784)
    db1_grad = np.sum(delta1, axis = 0, keepdims = True).T    # (512, 1)
    
    return [dj_dw1, dj_dw2, dj_dw3, db1_grad, db2_grad, db3_grad]


# Complete the below function to complete the backpropagation ste
def backPropagate_2(inputs, bias, targets, weights, activations, act_func, learning_rate):
    # Inputs: input data, targets, parameters of network, intermediate activations, learning rate of optimization algorithm
    gradients = computeGradients_2(inputs, targets, weights, activations, act_func)
    
    # update weights
    weights, bias = applyGradients(weights, bias, gradients, learning_rate)
    
    return weights, bias

In [29]:
# Complete the below function to complete the training of network
def training_2(inputs, targets, act_func, batch_size = 120, epochs = 30, train_val_split = 0.8, learning_rate = 0.0002):
    # set hyperparameters
    hidden_units = 512
    n_classes = 10
    n_samples = inputs.shape[0]
    
    # Split the training data into two parts.
    # Use 90 percent of training data for training the network.
    # Remaining 10 percent as validation data
    split_idx = int(train_val_split * n_samples)
    train_inputs, train_targets = inputs[ : split_idx], targets[ : split_idx]
    test_inputs, test_targets = inputs[split_idx : ], targets[split_idx : ]
    
    n_batches = split_idx // batch_size
    
    # Randomly initialize weights
    W1 = np.random.randn(hidden_units, inputs.shape[1]) * 0.01
    b1 = np.random.uniform(0, 1, (hidden_units, 1))
    W2 = np.random.randn(hidden_units, hidden_units) * 0.01
    b2 = np.random.uniform(0, 1, (hidden_units, 1))
    W3 = np.random.randn(n_classes, hidden_units) * 0.01
    b3 = np.random.uniform(0, 1, (n_classes, 1))

    weights = [W1, W2, W3]
    bias = [b1, b2, b3]
    
    # Interate for epochs times
    for epoch in range(epochs):
        # Shuffle the training data
        idxs = np.random.permutation(split_idx)
        train_inputs_shuffled = train_inputs[idxs]
        train_targets_shuffled = train_targets[idxs]
        
        # Iterate through the batches of data
        for batch in range(n_batches):
            # Get the batch of data
            st = batch * batch_size
            ed = min((batch + 1) * batch_size, split_idx)
            # print(st,ed)
            input_batch = train_inputs_shuffled[st : ed]
            target_batch = train_targets_shuffled[st : ed]
            
            # Forward propagation
            activations = fwdPropagate_2(input_batch, bias, weights, act_func)

            # Backward propagation
            weights, bias = backPropagate_2(input_batch, bias, target_batch, weights, activations, act_func, learning_rate)
            
        # Compute outpus on trianing data
        _, _, train_prob, _, _ = fwdPropagate_2(train_inputs, bias, weights, act_func)
        y1_pred = np.argmax(train_prob, axis = 1)
        y1_train = np.argmax(train_targets, axis = 1)
        train_acc, train_error = calc_error(y1_train, y1_pred)
        
        # Compute outpus on validation data
        _, _, test_prob, _, _ = fwdPropagate_2(test_inputs, bias, weights, act_func)
        y2_pred = np.argmax(test_prob, axis = 1)
        y2_test = np.argmax(test_targets, axis = 1)
        test_acc, test_error = calc_error(y2_test, y2_pred)
        
        # Print the statistics of training
        print(f"Epoch {epoch+1}/{epochs} - Training Error: {train_error:.4f} - Training Accuracy: {train_acc:.4f} - Validation Error: {test_error:.4f} - Validation Accuracy: {test_acc:.4f}") 
        
    return weights, bias

 
# Call the training function to train the network
print("For sigmoid activation function : ")
train_weights_sig, train_bias_sig = training_2(x_train_1, y_train_1, "sigmoid")
print('\n')

print("For tanh activation function : ")
train_weights_tanh, train_bias_tanh = training_2(x_train_1, y_train_1, "tanh")
print('\n')

print("For Leaky ReLu activation function : ")
train_weights_LReLu, train_bias_LReLu = training_2(x_train_1, y_train_1, "leakyrelu")
print('\n')

For sigmoid activation function : 
Epoch 1/30 - Training Error: 0.8860 - Training Accuracy: 0.1140 - Validation Error: 0.8940 - Validation Accuracy: 0.1060
Epoch 2/30 - Training Error: 0.8965 - Training Accuracy: 0.1035 - Validation Error: 0.8919 - Validation Accuracy: 0.1081
Epoch 3/30 - Training Error: 0.8981 - Training Accuracy: 0.1019 - Validation Error: 0.8965 - Validation Accuracy: 0.1035
Epoch 4/30 - Training Error: 0.8860 - Training Accuracy: 0.1140 - Validation Error: 0.8940 - Validation Accuracy: 0.1060
Epoch 5/30 - Training Error: 0.8860 - Training Accuracy: 0.1140 - Validation Error: 0.8940 - Validation Accuracy: 0.1060
Epoch 6/30 - Training Error: 0.9000 - Training Accuracy: 0.1000 - Validation Error: 0.9044 - Validation Accuracy: 0.0956
Epoch 7/30 - Training Error: 0.9011 - Training Accuracy: 0.0989 - Validation Error: 0.9025 - Validation Accuracy: 0.0975
Epoch 8/30 - Training Error: 0.9000 - Training Accuracy: 0.1000 - Validation Error: 0.9044 - Validation Accuracy: 0.09

- $\large{Evaluate\;\;the\;\;performance\;\;on\;\;MNIST\;\;test\;\;data}$

In [30]:
##################################################
#Evaluate the performance on MNIST test data
##################################################
def evaluate_performance_2(inputs, targets, weights, bias, act_func):
    # Forward propagate the inputs
    out = fwdPropagate_2(inputs, bias, weights, act_func)[2]
    
    # Assign input to the class having maximum posterior probability
    y_pred = np.argmax(out, axis = 1)
    y_train = np.argmax(targets, axis=1)
    
    # Compute accuracy
    acc, err = calc_error(y_train, y_pred)
    
    return acc

print("For sigmoid activation function : ")
test_accuracy_sig = evaluate_performance_2(x_test_1, y_test_1, train_weights_sig, train_bias_sig, "sigmoid")
print(f"Test Accuracy : {test_accuracy_sig}\n")

print("For tanh activation function : ")
test_accuracy_tanh = evaluate_performance_2(x_test_1, y_test_1, train_weights_tanh, train_bias_tanh, "tanh")
print(f"Test Accuracy : {test_accuracy_tanh}\n")

print("For Leaky ReLu activation function : ")
test_accuracy_LReLu = evaluate_performance_2(x_test_1, y_test_1, train_weights_LReLu, train_bias_LReLu, "leakyrelu")
print("Test Accuracy:", test_accuracy_LReLu)

For sigmoid activation function : 
Test Accuracy : 0.101

For tanh activation function : 
Test Accuracy : 0.951

For Leaky ReLu activation function : 
Test Accuracy: 0.9631


<b> Report your observations </b>

1. Sigmoid Activation Function:

- The training and validation errors fluctuate around 0.89, indicating poor convergence.
- The training and validation accuracies remain low, around 10%.
The test accuracy is also very low, at around 10.1%.
- Sigmoid activation may not be suitable for this task, as it suffers from the vanishing gradient problem and struggles to capture complex patterns.

2. Tanh Activation Function:

- The training and validation errors decrease consistently over epochs, indicating good convergence.
Both training and validation accuracies increase steadily and reach around 92-93%.
- The test accuracy is relatively high, around 95.1%, suggesting good generalization.
- Tanh activation function is performing better than sigmoid, as it avoids the vanishing gradient problem and allows for better gradient flow.

3. Leaky ReLU Activation Function:

- The training and validation errors decrease consistently over epochs, similar to tanh.
Both training and validation accuracies increase steadily and reach around 96-97%.
- The test accuracy is the highest among the three activation functions, at around 96.31%.
- Leaky ReLU performs the best among the three activation functions, indicating that it allows for faster convergence and better capture of complex patterns.



<b> Part - (3) : Understanding optimization algorithms: </b> In this part you will learn to use different optimiztion algorithm apart from SGD.

- Using the best activation function from Part - (2), train the classification network using Adam optimization algorithm. 

- Compare the accuracy of the networks trained with SGD and Adam optimization algorithms. 

- Report your observations. 

- $\large{Train\;\;the\;\;network\;\;using\;\;Adam\;\;optimizer}$

In [31]:
# Complete the below function to update the parameters using the above computed gradients
def applyGradients_adams_opt(weights, bias, gradients, learning_rate, beta1 = 0.9, beta2 = 0.999, epsilon = 1e-8):
    # Inputs: weights, gradients, and learning rate
    W1, W2, W3 = weights        # W1 (512, 784), W2 (512, 512), W3 (10, 512)
    b1, b2, b3 = bias
    dj_dw1, dj_dw2, dj_dw3, db1_grad, db2_grad, db3_grad = gradients
    
    # Initialize adam parameters
    m_dW1, m_dW2, m_dW3 = np.zeros(W1.shape), np.zeros(W2.shape), np.zeros(W3.shape)
    v_dW1, v_dW2, v_dW3 = np.zeros(W1.shape), np.zeros(W2.shape), np.zeros(W3.shape)
    m_db1, m_db2, m_db3 = np.zeros(b1.shape), np.zeros(b2.shape), np.zeros(b3.shape)
    v_db1, v_db2, v_db3 = np.zeros(b1.shape), np.zeros(b2.shape), np.zeros(b3.shape)
    
    # Bias correction terms initialization
    m_crct_dW1, m_crct_dW2, m_crct_dW3 = np.zeros(W1.shape), np.zeros(W2.shape), np.zeros(W3.shape)
    m_crct_db1, m_crct_db2, m_crct_db3 = np.zeros(b1.shape), np.zeros(b2.shape), np.zeros(b3.shape)
    v_crct_dW1, v_crct_dW2, v_crct_dW3 = np.zeros(W1.shape), np.zeros(W2.shape), np.zeros(W3.shape)
    v_crct_db1, v_crct_db2, v_crct_db3 = np.zeros(b1.shape), np.zeros(b2.shape), np.zeros(b3.shape)
    
    params = [W1, W2, W3, b2, b2, b3]
    grads = [dj_dw1, dj_dw2, dj_dw3, db1_grad, db2_grad, db3_grad]
    m_params = [m_dW1, m_dW2, m_dW3, m_db1, m_db2, m_db3]
    v_params = [v_dW1, v_dW2, v_dW3, v_db1, v_db2, v_db3]
    m_crct_params = [m_crct_dW1, m_crct_dW2, m_crct_dW3, m_crct_db1, m_crct_db2, m_crct_db3]
    v_crct_params = [v_crct_dW1, v_crct_dW2, v_crct_dW3, v_crct_db1, v_crct_db2, v_crct_db3]
        
    # Adam update
    t = 0
    for i in range(6):
        t += 1
        m_params[i] = beta1 * m_params[i] + (1 - beta1) * grads[i]
        v_params[i] = beta2 * v_params[i] + (1 - beta2) * (grads[i] ** 2)
        
        m_crct_params[i] = m_params[i] / (1 - beta1 ** t)
        v_crct_params[i] = v_params[i] / (1 - beta2 ** t)
        
        params[i] = params[i] - learning_rate * m_crct_params[i] / (np.sqrt(v_crct_params[i]) + epsilon)
    
    return params[ : 3], params[3 : ]

# Complete the below function to complete the backpropagation ste
def backPropagate_adams_opt(inputs, bias, targets, weights, activations, act_func, learning_rate):
    # Inputs: input data, targets, parameters of network, intermediate activations, learning rate of optimization algorithm
    gradients = computeGradients_2(inputs, targets, weights, activations, act_func)
    
    # update weights
    weights, bias = applyGradients_adams_opt(weights, bias, gradients, learning_rate)
    
    return weights, bias

In [32]:
# Complete the below function to complete the training of network
def training_adams_opt(inputs, targets, act_func, batch_size = 120, epochs = 30, train_val_split = 0.8, learning_rate = 0.0002):
    # set hyperparameters
    hidden_units = 512
    n_classes = 10
    n_samples = inputs.shape[0]
    
    # Split the training data into two parts.
    # Use 90 percent of training data for training the network.
    # Remaining 10 percent as validation data
    split_idx = int(train_val_split * n_samples)
    train_inputs, train_targets = inputs[ : split_idx], targets[ : split_idx]
    test_inputs, test_targets = inputs[split_idx : ], targets[split_idx : ]
    
    n_batches = split_idx // batch_size
    
    # Randomly initialize weights
    W1 = np.random.randn(hidden_units, inputs.shape[1]) * 0.01
    b1 = np.random.uniform(0, 1, (hidden_units, 1))
    W2 = np.random.randn(hidden_units, hidden_units) * 0.01
    b2 = np.random.uniform(0, 1, (hidden_units, 1))
    W3 = np.random.randn(n_classes, hidden_units) * 0.01
    b3 = np.random.uniform(0, 1, (n_classes, 1))

    weights = [W1, W2, W3]
    bias = [b1, b2, b3]
    
    # Interate for epochs times
    for epoch in range(epochs):
        # Shuffle the training data
        idxs = np.random.permutation(split_idx)
        train_inputs_shuffled = train_inputs[idxs]
        train_targets_shuffled = train_targets[idxs]
        
        # Iterate through the batches of data
        for batch in range(n_batches):
            # Get the batch of data
            st = batch * batch_size
            ed = min((batch + 1) * batch_size, split_idx)
            # print(st,ed)
            input_batch = train_inputs_shuffled[st : ed]
            target_batch = train_targets_shuffled[st : ed]
            
            # Forward propagation
            activations = fwdPropagate_2(input_batch, bias, weights, act_func)

            # Backward propagation
            weights, bias = backPropagate_adams_opt(input_batch, bias, target_batch, weights, activations, act_func, learning_rate)
            
        # Compute outpus on trianing data
        _, _, train_prob, _, _ = fwdPropagate_2(train_inputs, bias, weights, act_func)
        y1_pred = np.argmax(train_prob, axis = 1)
        y1_train = np.argmax(train_targets, axis = 1)
        train_acc, train_error = calc_error(y1_train, y1_pred)
        
        # Compute outpus on validation data
        _, _, test_prob, _, _ = fwdPropagate_2(test_inputs, bias, weights, act_func)
        y2_pred = np.argmax(test_prob, axis = 1)
        y2_test = np.argmax(test_targets, axis = 1)
        test_acc, test_error = calc_error(y2_test, y2_pred)
        
        # Print the statistics of training
        print(f"Epoch {epoch+1}/{epochs} - Training Error: {train_error:.4f} - Training Accuracy: {train_acc:.4f} - Validation Error: {test_error:.4f} - Validation Accuracy: {test_acc:.4f}") 
        
    return weights, bias

 
# Call the training function to train the network
print("For Leaky ReLu activation function : ")
train_weights_LReLu, train_bias_LReLu = training_2(x_train_1, y_train_1, "leakyrelu")
print('\n')

For Leaky ReLu activation function : 
Epoch 1/30 - Training Error: 0.3702 - Training Accuracy: 0.6298 - Validation Error: 0.3546 - Validation Accuracy: 0.6454
Epoch 2/30 - Training Error: 0.1734 - Training Accuracy: 0.8266 - Validation Error: 0.1626 - Validation Accuracy: 0.8374
Epoch 3/30 - Training Error: 0.1262 - Training Accuracy: 0.8738 - Validation Error: 0.1192 - Validation Accuracy: 0.8808
Epoch 4/30 - Training Error: 0.1119 - Training Accuracy: 0.8881 - Validation Error: 0.1091 - Validation Accuracy: 0.8909
Epoch 5/30 - Training Error: 0.1014 - Training Accuracy: 0.8986 - Validation Error: 0.0977 - Validation Accuracy: 0.9023
Epoch 6/30 - Training Error: 0.0952 - Training Accuracy: 0.9048 - Validation Error: 0.0917 - Validation Accuracy: 0.9083
Epoch 7/30 - Training Error: 0.0915 - Training Accuracy: 0.9085 - Validation Error: 0.0887 - Validation Accuracy: 0.9113
Epoch 8/30 - Training Error: 0.0858 - Training Accuracy: 0.9142 - Validation Error: 0.0821 - Validation Accuracy: 0

In [33]:
##################################################
#Compare the accuracies and report your observations
##################################################
print("For Leaky ReLu activation function : ")
test_accuracy_LReLu = evaluate_performance_2(x_test_1, y_test_1, train_weights_LReLu, train_bias_LReLu, "leakyrelu")
print("Test Accuracy:", test_accuracy_LReLu)

For Leaky ReLu activation function : 
Test Accuracy: 0.9629


<b> Report your observations </b>

1. Training using adams optimization technique, the Leaky ReLU activation function shows a gradual decrease in both training and validation errors across epochs, indicating effective learning.

2. The training accuracy increases steadily with epochs, reaching around 97% by the end of training, suggesting that the model is learning the patterns in the training data well.

3. The validation accuracy also increases consistently, reaching around 96% by the end of training. This suggests that the model is generalizing well to unseen data.

4. The final test accuracy achieved using the trained model on the test data is approximately 96.29%, indicating good performance on unseen data as well.



<b> Part - (4) : Understanding regularization methods: </b> In this part of the assignment, you will learn about a few regularization techniques to reduce the overfitting problem. Using the above built network, include the following techniques to reduce the overfitting by retraining the network efficiently. Write down the accuracies for each case.

- Weight regularization: Add regularization term to the classification loss 

- Dropout with a probability of 0.2: Randomly drop the activation potentials of hidden neural with 0.2 probability. Disable the dropout layer in inference model. You can experiment with different dropout probabilities and report your observations.

- Early stopping: Stop the network training when it is started to overfitting to training data. 


- $\large{Training\;\;with\;\;weight\;\;regularization}$

In [35]:
# Complete the below function to compute the gradients
def computeGradients_4(inputs, targets, weights, activations, act_func, reg_lam = 0.01):
    # Inputs: input data, targets, parameters of netwrok, intermediate activations
    W1, W2, W3 = weights        # W1 (512, 784), W2 (512, 512), W3 (10, 512)

    # calculate activations
    z1, z2, y, a1, a2 = activations     # a1 (batch_size, 512), a2 (batch_size, 512), y (batch_size, 10)
    
    # compute the loss
    
    # Compute the derivative of loss at parameters
    delta3 = y - targets        # delta3 (batch_size, 10)
    dj_dw3 = delta3.T @ z2 + reg_lam * W3    # (10, 512)
    db3_grad = np.sum(delta3, axis = 0, keepdims = True).T    # (10, 1)

    # Hidden layer 2
    da2 = gradSigmoid(a2)
    if (act_func == "tanh"):
        da2 = gradtanH(a2)
    elif (act_func == "leakyrelu"):
        da2 = gradLeakyReLu(a2)
        
    delta2 = (delta3 @ W3) * da2        # delta3 (batch_size, 512)
    dj_dw2 = delta2.T @ z1 + reg_lam * W2    # (512, 512)
    db2_grad = np.sum(delta2, axis = 0, keepdims = True).T    # (10, 1)
    
    # Hidden layer 1
    da1 = gradSigmoid(a1)
    if (act_func == "tanh"):
        da1 = gradtanH(a1)
    elif (act_func == "leakyrelu"):
        da1 = gradLeakyReLu(a1)
        
    delta1 = (delta2 @ W2) * da1        # delta3 (batch_size, 10)
    dj_dw1 = delta1.T @ inputs + reg_lam * W1    # (512, 784)
    db1_grad = np.sum(delta1, axis = 0, keepdims = True).T    # (512, 1)
    
    return [dj_dw1, dj_dw2, dj_dw3, db1_grad, db2_grad, db3_grad]

# Complete the below function to complete the backpropagation ste
def backPropagate_4(inputs, bias, targets, weights, activations, act_func, learning_rate):
    # Inputs: input data, targets, parameters of network, intermediate activations, learning rate of optimization algorithm
    gradients = computeGradients_4(inputs, targets, weights, activations, act_func)
    
    # update weights
    weights, bias = applyGradients(weights, bias, gradients, learning_rate)
    
    return weights, bias

# Complete the below function to complete the training of network
def training_4(inputs, targets, act_func, batch_size = 120, epochs = 30, train_val_split = 0.8, learning_rate = 0.0002):
    # set hyperparameters
    hidden_units = 512
    n_classes = 10
    n_samples = inputs.shape[0]
    
    # Split the training data into two parts.
    # Use 90 percent of training data for training the network.
    # Remaining 10 percent as validation data
    split_idx = int(train_val_split * n_samples)
    train_inputs, train_targets = inputs[ : split_idx], targets[ : split_idx]
    test_inputs, test_targets = inputs[split_idx : ], targets[split_idx : ]
    
    n_batches = split_idx // batch_size
    
    # Randomly initialize weights
    W1 = np.random.randn(hidden_units, inputs.shape[1]) * 0.01
    b1 = np.random.uniform(0, 1, (hidden_units, 1))
    W2 = np.random.randn(hidden_units, hidden_units) * 0.01
    b2 = np.random.uniform(0, 1, (hidden_units, 1))
    W3 = np.random.randn(n_classes, hidden_units) * 0.01
    b3 = np.random.uniform(0, 1, (n_classes, 1))

    weights = [W1, W2, W3]
    bias = [b1, b2, b3]
    
    # Interate for epochs times
    for epoch in range(epochs):
        # Shuffle the training data
        idxs = np.random.permutation(split_idx)
        train_inputs_shuffled = train_inputs[idxs]
        train_targets_shuffled = train_targets[idxs]
        
        # Iterate through the batches of data
        for batch in range(n_batches):
            # Get the batch of data
            st = batch * batch_size
            ed = min((batch + 1) * batch_size, split_idx)
            # print(st,ed)
            input_batch = train_inputs_shuffled[st : ed]
            target_batch = train_targets_shuffled[st : ed]
            
            # Forward propagation
            activations = fwdPropagate_2(input_batch, bias, weights, act_func)

            # Backward propagation
            weights, bias = backPropagate_4(input_batch, bias, target_batch, weights, activations, act_func, learning_rate)
            
        # Compute outpus on trianing data
        _, _, train_prob, _, _ = fwdPropagate_2(train_inputs, bias, weights, act_func)
        y1_pred = np.argmax(train_prob, axis = 1)
        y1_train = np.argmax(train_targets, axis = 1)
        train_acc, train_error = calc_error(y1_train, y1_pred)
        
        # Compute outpus on validation data
        _, _, test_prob, _, _ = fwdPropagate_2(test_inputs, bias, weights, act_func)
        y2_pred = np.argmax(test_prob, axis = 1)
        y2_test = np.argmax(test_targets, axis = 1)
        test_acc, test_error = calc_error(y2_test, y2_pred)
        
        # Print the statistics of training
        print(f"Epoch {epoch+1}/{epochs} - Training Error: {train_error:.4f} - Training Accuracy: {train_acc:.4f} - Validation Error: {test_error:.4f} - Validation Accuracy: {test_acc:.4f}") 
        
    return weights, bias

 
# Call the training function to train the network
print("For Leaky ReLu activation function with regularization : ")
train_weights_LReLu_4, train_bias_LReLu_4 = training_2(x_train_1, y_train_1, "leakyrelu")
print('\n')

print("For Leaky ReLu activation function with regularization : ")
test_accuracy_LReLu_4 = evaluate_performance_2(x_test_1, y_test_1, train_weights_LReLu_4, train_bias_LReLu_4, "leakyrelu")
print("Test Accuracy:", test_accuracy_LReLu_4)

For Leaky ReLu activation function with regularization : 
Epoch 1/30 - Training Error: 0.4157 - Training Accuracy: 0.5843 - Validation Error: 0.4007 - Validation Accuracy: 0.5993
Epoch 2/30 - Training Error: 0.1690 - Training Accuracy: 0.8310 - Validation Error: 0.1581 - Validation Accuracy: 0.8419
Epoch 3/30 - Training Error: 0.1259 - Training Accuracy: 0.8741 - Validation Error: 0.1198 - Validation Accuracy: 0.8802
Epoch 4/30 - Training Error: 0.1096 - Training Accuracy: 0.8904 - Validation Error: 0.1056 - Validation Accuracy: 0.8944
Epoch 5/30 - Training Error: 0.1017 - Training Accuracy: 0.8983 - Validation Error: 0.0986 - Validation Accuracy: 0.9014
Epoch 6/30 - Training Error: 0.0962 - Training Accuracy: 0.9038 - Validation Error: 0.0933 - Validation Accuracy: 0.9067
Epoch 7/30 - Training Error: 0.0915 - Training Accuracy: 0.9085 - Validation Error: 0.0881 - Validation Accuracy: 0.9119
Epoch 8/30 - Training Error: 0.0894 - Training Accuracy: 0.9106 - Validation Error: 0.0855 - Va

- $\large{Training\;\;with\;\;dropout\;\;strategy}$

In [55]:
# Complete the below function to impliment forward propagation of data
def fwdPropagate_dropout(inputs, bias, weights, act_func, p):
    # Inputs: input data, paramters of network
    W1, W2, W3 = weights    # W1 (512, 784), W2 (512, 512), W3 (10, 512)
    b1, b2, b3 = bias
    
    # compute a1
    a1 = inputs @ W1.T + b1.T  # (batch_size, 512)
    
    # compute z1
    z1 = sigmoid(a1)
    if (act_func == "tanh"):
        z1 = tanH(a1)
    elif (act_func == "leakyrelu"):
        z1 = LeakyReLu(a1)
        
    # compute mask1
    mask1 = np.random.binomial(1, 1 - p, size = z1.shape)
    z1 = z1 * mask1
    
    # compute a2  
    a2 = z1 @ W2.T + b2.T    # (batch_size, 512)
    
    # compute z2
    z2 = sigmoid(a2)
    if (act_func == "tanh"):
        z2 = tanH(a2)
    elif (act_func == "leakyrelu"):
        z2 = LeakyReLu(a2)
        
    # compute mask1
    mask2 = np.random.binomial(1, 1 - p, size = z2.shape)
    z2 = z2 * mask2
    
    # compute a3 and y
    a3 = z2 @ W3.T + b3.T      # (batch_size, 10)
    y = softmax(a3)
    
    return [z1, z2, y, a1, a2], [mask1, mask2]

# Complete the below function to compute the gradients
def computeGradients_dropout(inputs, targets, weights, activations, mask, act_func):
    # Inputs: input data, targets, parameters of netwrok, intermediate activations
    W1, W2, W3 = weights        # W1 (512, 784), W2 (512, 512), W3 (10, 512)

    # calculate activations
    z1, z2, y, a1, a2 = activations     # a1 (batch_size, 512), a2 (batch_size, 512), y (batch_size, 10)
    mask1, mask2 = mask
    
    # compute the loss
    
    # Compute the derivative of loss at parameters
    delta3 = y - targets        # delta3 (batch_size, 10)
    dj_dw3 = delta3.T @ z2     # (10, 512)
    db3_grad = np.sum(delta3, axis = 0, keepdims = True).T    # (10, 1)

    # Hidden layer 2
    da2 = gradSigmoid(a2)
    if (act_func == "tanh"):
        da2 = gradtanH(a2)
    elif (act_func == "leakyrelu"):
        da2 = gradLeakyReLu(a2)
        
    delta2 = (delta3 @ W3) * da2 * mask2       # delta3 (batch_size, 512)
    dj_dw2 = delta2.T @ z1     # (512, 512)
    db2_grad = np.sum(delta2, axis = 0, keepdims = True).T    # (10, 1)
    
    # Hidden layer 1
    da1 = gradSigmoid(a1)
    if (act_func == "tanh"):
        da1 = gradtanH(a1)
    elif (act_func == "leakyrelu"):
        da1 = gradLeakyReLu(a1)
        
    delta1 = (delta2 @ W2) * da1 * mask1       # delta3 (batch_size, 10)
    dj_dw1 = delta1.T @ inputs     # (512, 784)
    db1_grad = np.sum(delta1, axis = 0, keepdims = True).T    # (512, 1)
    
    return [dj_dw1, dj_dw2, dj_dw3, db1_grad, db2_grad, db3_grad]


# Complete the below function to complete the backpropagation ste
def backPropagate_dropout(inputs, bias, targets, weights, activations, mask, act_func, learning_rate):
    # Inputs: input data, targets, parameters of network, intermediate activations, learning rate of optimization algorithm
    gradients = computeGradients_dropout(inputs, targets, weights, activations, mask, act_func)
    
    # update weights
    weights, bias = applyGradients(weights, bias, gradients, learning_rate)
    
    return weights, bias

In [59]:
# Complete the below function to complete the training of network
def training_dropout(inputs, targets, act_func, p, batch_size = 120, epochs = 30, train_val_split = 0.8, learning_rate = 0.0002):
    # set hyperparameters
    hidden_units = 512
    n_classes = 10
    n_samples = inputs.shape[0]
    
    # Split the training data into two parts.
    # Use 90 percent of training data for training the network.
    # Remaining 10 percent as validation data
    split_idx = int(train_val_split * n_samples)
    train_inputs, train_targets = inputs[ : split_idx], targets[ : split_idx]
    test_inputs, test_targets = inputs[split_idx : ], targets[split_idx : ]
    
    n_batches = split_idx // batch_size
    
    # Randomly initialize weights
    W1 = np.random.randn(hidden_units, inputs.shape[1]) * 0.01
    b1 = np.random.uniform(0, 1, (hidden_units, 1))
    W2 = np.random.randn(hidden_units, hidden_units) * 0.01
    b2 = np.random.uniform(0, 1, (hidden_units, 1))
    W3 = np.random.randn(n_classes, hidden_units) * 0.01
    b3 = np.random.uniform(0, 1, (n_classes, 1))

    weights = [W1, W2, W3]
    bias = [b1, b2, b3]
    
    # Interate for epochs times
    for epoch in range(epochs):
        # Shuffle the training data
        idxs = np.random.permutation(split_idx)
        train_inputs_shuffled = train_inputs[idxs]
        train_targets_shuffled = train_targets[idxs]
        
        # Iterate through the batches of data
        for batch in range(n_batches):
            # Get the batch of data
            st = batch * batch_size
            ed = min((batch + 1) * batch_size, split_idx)
            # print(st,ed)
            input_batch = train_inputs_shuffled[st : ed]
            target_batch = train_targets_shuffled[st : ed]
            
            # Forward propagation
            activations, mask = fwdPropagate_dropout(input_batch, bias, weights, act_func, p)

            # Backward propagation
            weights, bias = backPropagate_dropout(input_batch, bias, target_batch, weights, activations, mask, act_func, learning_rate)
            
        # Compute outpus on trianing data
        train_prob, _ = fwdPropagate_dropout(train_inputs, bias, weights, act_func, p)
        y1_pred = np.argmax(train_prob[2], axis = 1)
        y1_train = np.argmax(train_targets, axis = 1)
        train_acc, train_error = calc_error(y1_train, y1_pred)
        
        # Compute outpus on validation data
        test_prob, _ = fwdPropagate_dropout(test_inputs, bias, weights, act_func, p)
        y2_pred = np.argmax(test_prob[2], axis = 1)
        y2_test = np.argmax(test_targets, axis = 1)
        test_acc, test_error = calc_error(y2_test, y2_pred)
        
        # Print the statistics of training
        print(f"Epoch {epoch+1}/{epochs} - Training Error: {train_error:.4f} - Training Accuracy: {train_acc:.4f} - Validation Error: {test_error:.4f} - Validation Accuracy: {test_acc:.4f}") 
        
    return weights, bias

 
# Call the training function to train the network
print("For Leaky ReLu activation function with dropout : ")
train_weights_LReLu_dropout, train_bias_LReLu_dropout = training_dropout(x_train_1, y_train_1, "leakyrelu", 0.2)
print('\n')

For Leaky ReLu activation function with dropout : 
Epoch 1/30 - Training Error: 0.7438 - Training Accuracy: 0.2562 - Validation Error: 0.7524 - Validation Accuracy: 0.2476
Epoch 2/30 - Training Error: 0.3474 - Training Accuracy: 0.6526 - Validation Error: 0.3335 - Validation Accuracy: 0.6665
Epoch 3/30 - Training Error: 0.2107 - Training Accuracy: 0.7893 - Validation Error: 0.1977 - Validation Accuracy: 0.8023
Epoch 4/30 - Training Error: 0.1598 - Training Accuracy: 0.8402 - Validation Error: 0.1502 - Validation Accuracy: 0.8498
Epoch 5/30 - Training Error: 0.1365 - Training Accuracy: 0.8635 - Validation Error: 0.1291 - Validation Accuracy: 0.8709
Epoch 6/30 - Training Error: 0.1215 - Training Accuracy: 0.8785 - Validation Error: 0.1177 - Validation Accuracy: 0.8823
Epoch 7/30 - Training Error: 0.1138 - Training Accuracy: 0.8862 - Validation Error: 0.1121 - Validation Accuracy: 0.8879
Epoch 8/30 - Training Error: 0.1082 - Training Accuracy: 0.8918 - Validation Error: 0.1026 - Validatio

In [60]:
def evaluate_performance_dropout(inputs, targets, weights, bias, act_func, p):
    # Forward propagate the inputs
    activ, mask = fwdPropagate_dropout(inputs, bias, weights, act_func, p)
    
    # Assign input to the class having maximum posterior probability
    y_pred = np.argmax(activ[2], axis = 1)
    y_train = np.argmax(targets, axis=1)
    
    # Compute accuracy
    acc, err = calc_error(y_train, y_pred)
    
    return acc

print("For Leaky ReLu activation function : ")
test_accuracy_LReLu_dropout = evaluate_performance_dropout(x_test_1, y_test_1, train_weights_LReLu_dropout, train_bias_LReLu_dropout, "leakyrelu", 0.2)
print("Test Accuracy:", test_accuracy_LReLu_dropout)

For Leaky ReLu activation function : 
Test Accuracy: 0.9518


- $\large{Training\;\;with\;\;early\;\;stopping\;\;criterion}$

In [64]:
# Complete the below function to complete the training of network
def training_earlystopping(inputs, targets, act_func, batch_size = 120, epochs = 30, train_val_split = 0.8, learning_rate = 0.0002):
    # set hyperparameters
    hidden_units = 512
    n_classes = 10
    n_samples = inputs.shape[0]
    
    # Split the training data into two parts.
    # Use 90 percent of training data for training the network.
    # Remaining 10 percent as validation data
    split_idx = int(train_val_split * n_samples)
    train_inputs, train_targets = inputs[ : split_idx], targets[ : split_idx]
    test_inputs, test_targets = inputs[split_idx : ], targets[split_idx : ]
    
    n_batches = split_idx // batch_size
    
    # Randomly initialize weights
    W1 = np.random.randn(hidden_units, inputs.shape[1]) * 0.01
    b1 = np.random.uniform(0, 1, (hidden_units, 1))
    W2 = np.random.randn(hidden_units, hidden_units) * 0.01
    b2 = np.random.uniform(0, 1, (hidden_units, 1))
    W3 = np.random.randn(n_classes, hidden_units) * 0.01
    b3 = np.random.uniform(0, 1, (n_classes, 1))

    weights = [W1, W2, W3]
    bias = [b1, b2, b3]
    
    # best_val_err = - 1.0
    epochs_since_best_err = 0
    best_weights = None
    best_bias = None
    
    # Interate for epochs times
    for epoch in range(epochs):
        
        if epochs_since_best_err >= 3:
            print("Early stopping due to overfitting")
            break
        
        # Shuffle the training data
        idxs = np.random.permutation(split_idx)
        train_inputs_shuffled = train_inputs[idxs]
        train_targets_shuffled = train_targets[idxs]
        
        # Iterate through the batches of data
        for batch in range(n_batches):
            # Get the batch of data
            st = batch * batch_size
            ed = min((batch + 1) * batch_size, split_idx)
            # print(st,ed)
            input_batch = train_inputs_shuffled[st : ed]
            target_batch = train_targets_shuffled[st : ed]
            
            # Forward propagation
            activations = fwdPropagate_2(input_batch, bias, weights, act_func)

            # Backward propagation
            weights, bias = backPropagate_2(input_batch, bias, target_batch, weights, activations, act_func, learning_rate)
            
        # Compute outpus on trianing data
        _, _, train_prob, _, _ = fwdPropagate_2(train_inputs, bias, weights, act_func)
        y1_pred = np.argmax(train_prob, axis = 1)
        y1_train = np.argmax(train_targets, axis = 1)
        train_acc, train_error = calc_error(y1_train, y1_pred)
        
        # Compute outpus on validation data
        _, _, test_prob, _, _ = fwdPropagate_2(test_inputs, bias, weights, act_func)
        y2_pred = np.argmax(test_prob, axis = 1)
        y2_test = np.argmax(test_targets, axis = 1)
        test_acc, test_error = calc_error(y2_test, y2_pred)
        
        # Print the statistics of training
        print(f"Epoch {epoch+1}/{epochs} - Training Error: {train_error:.4f} - Training Accuracy: {train_acc:.4f} - Validation Error: {test_error:.4f} - Validation Accuracy: {test_acc:.4f}") 
        
        # Check for early stopping
        if (test_error < train_error) :
            epochs_since_best_err = 0
            best_weights = weights.copy()
            best_bias = bias.copy()
        else:
            epochs_since_best_err += 1
        
    return best_weights, best_bias

 
# Call the training function to train the network
print("For Leaky ReLu activation function with earlystopping : ")
train_weights_LReLu_earlystopping, train_bias_LReLu_earlystopping = training_earlystopping(x_train_1, y_train_1, "leakyrelu")
print('\n')

For Leaky ReLu activation function with earlystopping : 
Epoch 1/30 - Training Error: 0.4780 - Training Accuracy: 0.5220 - Validation Error: 0.4658 - Validation Accuracy: 0.5342
Epoch 2/30 - Training Error: 0.1789 - Training Accuracy: 0.8211 - Validation Error: 0.1649 - Validation Accuracy: 0.8351
Epoch 3/30 - Training Error: 0.1264 - Training Accuracy: 0.8736 - Validation Error: 0.1181 - Validation Accuracy: 0.8819
Epoch 4/30 - Training Error: 0.1121 - Training Accuracy: 0.8879 - Validation Error: 0.1098 - Validation Accuracy: 0.8902
Epoch 5/30 - Training Error: 0.1035 - Training Accuracy: 0.8965 - Validation Error: 0.0988 - Validation Accuracy: 0.9012
Epoch 6/30 - Training Error: 0.0954 - Training Accuracy: 0.9046 - Validation Error: 0.0920 - Validation Accuracy: 0.9080
Epoch 7/30 - Training Error: 0.0901 - Training Accuracy: 0.9099 - Validation Error: 0.0855 - Validation Accuracy: 0.9145
Epoch 8/30 - Training Error: 0.0857 - Training Accuracy: 0.9143 - Validation Error: 0.0823 - Val

In [66]:
print("For Leaky ReLu activation function : ")
test_accuracy_LReLu_earlystopping = evaluate_performance_2(x_test_1, y_test_1, train_weights_LReLu_earlystopping, train_bias_LReLu_earlystopping, "leakyrelu")
print("Test Accuracy:", test_accuracy_LReLu_earlystopping)

For Leaky ReLu activation function : 
Test Accuracy: 0.949


<b> Report your observations </b>

1. Accuracy using regularization : 96.23 %

2. Accuracy using dropout : 95.18 %

3. Accuracy using earlystopping : 94.90 %

